# Deep Learning Project - 2025

### Università di Trento

**Authors:**  
- Antonio N. Bruno (ID: 258035)  
- Edoardo Di Tommaso (ID: 258433)



## Introduction

In recent years, large-scale Vision-Language Models (VLMs) such as CLIP [1] have demonstrated remarkable zero-shot classification performance by aligning images and text in a shared embedding space. This architecture enables flexible, prompt-driven recognition without any task-specific fine-tuning. However, when applied to fine-grained classification tasks, such as distinguishing between species of flowers, birds, or aircraft, the model’s performance tends to degrade, especially in low-data regimes.

This limitation has motivated growing interest in **few-shot adaptation**, where the goal is to adapt a pre-trained VLM to a new classification task using only a small number of labeled examples per class. In this setup, the model is trained on a few samples (shots) from a set of base classes, and then evaluated on both the base and a disjoint set of novel classes. The central challenge is to improve accuracy on base classes without sacrificing generalization to novel ones, a setting often referred to as **base-to-novel generalization**.

Our initial objective was to develop and test a few-shot adaptation method for CLIP ViT-B/16 on the Oxford Flowers dataset [2]. We experimented with several parameter-efficient fine-tuning (PEFT) approaches, including CoOp [3], CoCoOp [4], and KgCoOp[5]. However, during our investigation we uncovered a surprising finding: a substantial performance bottleneck was caused not by the model’s capacity, but by the **mismatch between dataset class names and CLIP’s training vocabulary**. By carefully aligning class labels with more natural or commonly used names, we significantly improved zero-shot accuracy, surpassing the gains obtained through fine-tuning. This suggests that label engineering can be as impactful as sophisticated adaptation methods.

In this notebook, we present the full workflow of our project: from baseline zero-shot evaluation with original class names, to systematic label alignment experiments, and finally to comparisons with state-of-the-art PEFT methods. Alongside code cells, we provide results, tables, figures, and discussion to make the report fully self-contained and reproducible.

### Imports and utilities for data handling

We will now create functions to correctly get our data and split it into base and novel classes.

In [ ]:
def get_data(data_dir="./data", transform=None):
    """Load Flowers102 train, validation and test sets.
    Args:
        data_dir (str): Directory where the dataset will be stored.
        transform (torch.Compose)
    Returns:
        tuple: A tuple containing the train, validation, and test sets.
    """
    train = torchvision.datasets.Flowers102(root=data_dir, split="train", download=True, transform=transform)
    val = torchvision.datasets.Flowers102(root=data_dir, split="val", download=True, transform=transform)
    test = torchvision.datasets.Flowers102(root=data_dir, split="test", download=True, transform=transform)
    return train, val, test

def base_novel_categories(dataset):
    # set returns the unique set of all dataset classes
    all_classes = set(dataset._labels)
    # and let's count them
    num_classes = len(all_classes)

    # here list(range(num_classes)) returns a list from 0 to num_classes - 1
    # then we slice the list in half and generate base and novel category lists
    base_classes = list(range(num_classes))[:num_classes//2]
    novel_classes = list(range(num_classes))[num_classes//2:]
    return base_classes, novel_classes

def split_data(dataset, base_classes):
    # these two lists will store the sample indexes
    base_categories_samples = []
    novel_categories_samples = []

    # we create a set of base classes to compute the test below in O(1)
    # this is optional and can be removed
    base_set = set(base_classes)

    # here we iterate over sample labels and also get the correspondent sample index
    for sample_id, label in enumerate(dataset._labels):
        if label in base_set:
            base_categories_samples.append(sample_id)
        else:
            novel_categories_samples.append(sample_id)

    # here we create the dataset subsets
    # the torch Subset is just a wrapper around the dataset
    # it simply stores the subset indexes and the original dataset (your_subset.dataset)
    # when asking for sample i in the subset, torch will look for its original position in the dataset and retrieve it
    # https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset
    base_dataset = torch.utils.data.Subset(dataset, base_categories_samples)
    novel_dataset = torch.utils.data.Subset(dataset, novel_categories_samples)
    return base_dataset, novel_dataset

def create_remapped_dataset(dataset, selected_classes):
    """Create a dataset subset with remapped labels.
    Args:
        dataset: Original dataset
        selected_classes: List of class indices to include
    Returns:
        subset dataset with labels remapped to [0, len(selected_classes)-1]
    """
    # Create mapping from original labels to new labels
    label_map = {old_label: new_label for new_label, old_label in enumerate(selected_classes)}
    selected_set = set(selected_classes)

    # Find samples and create new labels
    selected_samples = []
    new_labels = []

    for sample_id, label in enumerate(dataset._labels):
        if label in selected_set:
            selected_samples.append(sample_id)
            new_labels.append(label_map[label])

    # Create subset
    subset = torch.utils.data.Subset(dataset, selected_samples)

    # Add remapped labels to subset
    subset.remapped_labels = new_labels

    return subset

class RemappedDataset(torch.utils.data.Dataset):
    """Wrapper dataset that returns remapped labels"""
    def __init__(self, subset_dataset):
        self.dataset = subset_dataset.dataset
        self.indices = subset_dataset.indices
        self.labels = subset_dataset.remapped_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # Get original sample
        original_idx = self.indices[idx]
        image, _ = self.dataset[original_idx]  # Ignore original label

        # Return with remapped label
        return image, self.labels[idx]

## Baseline: CLIP Zero-Shot performance

We will now load the pretrained CLIP backbone and evaluate it on our dataset.

In [ ]:
# load CLIP. We will use Vit-B/16 as the visual backbone
device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, preprocess = clip.load("ViT-B/16", device=device) 
# preprocess contains CLIP's pre-defined augmentations

# define and inspect base and novel classes
_, _, tmp_test = get_data()
base_classes, novel_classes = base_novel_categories(tmp_test)
CLASS_NAMES = ["pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea",
                "english marigold", "tiger lily", "moon orchid", "bird of paradise", "monkshood",
                "globe thistle", "snapdragon", "colt's foot", "king protea", "spear thistle",
                "yellow iris", "globe-flower", "purple coneflower", "peruvian lily",
                "balloon flower", "giant white arum lily", "fire lily", "pincushion flower",
                "fritillary", "red ginger", "grape hyacinth", "corn poppy", "prince of wales feathers",
                "stemless gentian", "artichoke", "sweet william", "carnation", "garden phlox",
                "love in the mist", "mexican aster", "alpine sea holly", "ruby-lipped cattleya",
                "cape flower", "great masterwort", "siam tulip", "lenten rose", "barbeton daisy", "daffodil",
                "sword lily", "poinsettia", "bolero deep blue", "wallflower", "marigold", "buttercup",
                "oxeye daisy", "common dandelion", "petunia", "wild pansy", "primula", "sunflower",
                "pelargonium", "bishop of llandaff", "gaura", "geranium", "orange dahlia",
                "pink-yellow dahlia", "cautleya spicata", "japanese anemone", "black-eyed susan",
                "silverbush", "californian poppy", "osteospermum", "spring crocus", "bearded iris",
                "windflower", "tree poppy", "gazania", "azalea", "water lily", "rose", "thorn apple",
                "morning glory", "passion flower", "lotus", "toad lily", "anthurium", "frangipani",
                "clematis", "hibiscus", "columbine", "desert-rose", "tree mallow", "magnolia", "cyclamen",
                "watercress", "canna lily", "hippeastrum", "bee balm", "ball moss", "foxglove",
                "bougainvillea", "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower",
                "trumpet creeper", "blackberry lily"]
print("Base Class Names:", [(i, CLASS_NAMES[i]) for i in base_classes])
print("Novel Class Names:", [(i, CLASS_NAMES[i]) for i in novel_classes])

# get the three datasets
train_set, val_set, test_set = get_data(transform=preprocess)

# split classes into base and novel
base_classes, novel_classes = base_novel_categories(train_set)

# split the three datasets
train_base, _ = split_data(train_set, base_classes)
val_base, _ = split_data(val_set, base_classes)
test_base, test_novel = split_data(test_set, base_classes)

In [ ]:
# zero-shot predictions 

@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # here we apply the standard CLIP template used for oxford flowers to all categories
    # and immediately tokenize each sentence (convert natural language into numbers - feel free to print the text input to inspect them)
    text_inputs = clip.tokenize(
        [f"a photo of a {CLASS_NAMES[c]}, a type of flower." for c in categories]
    ).to(device)

    # we can encode the text features once as they are shared for all images
    # therefore we do it outside the evaluation loop
    text_features = model.encode_text(text_inputs)
    # and here we normalize them (standard pratice with CLIP)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    for image, target in tqdm(dataloader, desc=label):
        # base categories range from 0 to 50, whil novel ones from 51 to 101
        # therefore we must map categories to the [0, 50], otherwise we will have wrong predictions
        # Map targets in contiguous set starting from zero
        # Labels needs to be .long() in pytorch
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        # forward image through CLIP image encoder
        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # here cosine similarity between image and text features and keep the argmax for every row (every image)
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        # now we check which are correct, and sum them (False == 0, True == 1)
        correct_predictions += (predicted_class == target).sum().item()

    # and now we compute the accuracy
    accuracy = correct_predictions / len(dataset)
    return accuracy

base_accuracy = eval(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy = eval(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")

print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

def harmonic_mean(base_accuracy, novel_accuracy):
    numerator = 2
    denominator = 1 / base_accuracy + 1 / novel_accuracy
    hm = numerator / denominator
    return hm

print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")


## Double CLIP

One of the first ideas that came to mind was to enrich the prompt with context details related to the flower. So as first try in that direction we developd a systm that first used CLIP to infer the color of the given flower by classifying it within a determined range of colors, and then used the newfound information within the prompt for the specie classification.

In [ ]:
colors = [
"Red", "Light Pink", "Pink", "Dark Pink", "Magenta", "Orange", "Brown", "Yellow", "Green", "Blue", "Aqua", "Blue Gray", "Celeste", "Cyan", "Dark Blue", "Electric Blue",
"Light Blue", "Navy Blue", "Purple", "Violet", "Lilac", "Lavender", "Indigo", "Black", "Gray", "Light Gray", "White", "Creamy White", "Pearly White"
]

In [ ]:
@torch.no_grad()
def predict_colors(model, dataset, categories, batch_size, device, label=""):
    model.eval()

    # Create text inputs for colors
    color_text_inputs = clip.tokenize(
        [f"a photo of a {color} flower." for color in colors]
    ).to(device)

    # Encode text features for colors
    text_features = model.encode_text(color_text_inputs) 
    text_features /= text_features.norm(dim=-1, keepdim=True)

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    found_colors = []

    for image, target in tqdm(dataloader, desc=label):
        image = image.to(device)

        # Forward image through CLIP image encoder
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Get predicted color indices
        predicted_color_indices = (image_features @ text_features.T).argmax(dim=-1)

        # Convert indices to actual color names
        batch_colors = [colors[idx.item()] for idx in predicted_color_indices]
        found_colors.extend(batch_colors)  # Use extend to add all colors from batch

    return found_colors  # Return the list of predicted color names

found_colors = predict_colors(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="Finding color clues")
novel_found_colors = predict_colors(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="Finding color clues")
print("Color clues found")

In [ ]:
@torch.no_grad() # we don't want gradients
def eval(model, dataset, categories, batch_size, device, found_colors, label=""):
    # let's set the model in evaluation mode
    model.eval()

    # Remap labels into a contiguous set starting from zero
    contig_cat2idx = {cat: idx for idx, cat in enumerate(categories)}

    # simple dataloader creation
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # here we store the number of correct predictions we will make
    correct_predictions = 0
    sample_idx = 0  # Track which sample we're processing
    
    for image, target in tqdm(dataloader, desc=label):
        # Map targets in contiguous set starting from zero
        target = torch.Tensor([contig_cat2idx[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        image_features = model.encode_image(image)
        # and normalize
        image_features /= image_features.norm(dim=-1, keepdim=True)
        # For each image in the batch, create text prompts for all categories using that image's color
        batch_text_inputs = []
        for i in range(image.size(0)):
            if sample_idx+i < len(found_colors):
                sample_color = found_colors[sample_idx + i]
            else:
                sample_color = ""
            # Create prompts for all categories using this sample's color
            sample_prompts = [f"a photo of a {CLASS_NAMES[c]}, a {sample_color} colored type of flower." 
                            for c in categories]
            batch_text_inputs.extend(sample_prompts)
        
        # Tokenize all text inputs
        text_inputs = clip.tokenize(batch_text_inputs).to(device)
        # Reshape to (batch_size, num_categories, token_length)
        text_inputs = text_inputs.view(image.size(0), len(categories), -1)


        # Process each image in the batch
        batch_predictions = []
        for i in range(image.size(0)):
            # Get text features for all categories for this specific image
            sample_text_inputs = text_inputs[i]  # (num_categories, token_length)
            text_features = model.encode_text(sample_text_inputs)
            text_features /= text_features.norm(dim=-1, keepdim=True)
            
            # Compute similarity between this image and all category texts
            sample_image_features = image_features[i:i+1]  # Keep batch dimension
            similarities = (sample_image_features @ text_features.T)
            predicted_class = similarities.argmax(dim=-1)
            batch_predictions.append(predicted_class)
        
        predicted_classes = torch.cat(batch_predictions)
        
        # Check which predictions are correct
        correct_predictions += (predicted_classes == target).sum().item()
        
        # Update sample index
        sample_idx += image.size(0)

    # Compute accuracy
    accuracy = correct_predictions / len(dataset)
    return accuracy
# Updated function calls
base_accuracy = eval(model=clip_model, dataset=test_base, categories=base_classes, 
                    batch_size=64, device=device, found_colors=found_colors,
                    label="🧠 Zero-shot evaluation on Base Classes")

print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
novel_accuracy = eval(model=clip_model, dataset=test_novel, categories=novel_classes, 
                     batch_size=32, device=device, found_colors=novel_found_colors,
                     label="🧠 Zero-shot evaluation on Novel Classes")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")

In [ ]:
def harmonic_mean(base_accuracy, novel_accuracy):
    numerator = 2
    denominator = 1 / base_accuracy + 1 / novel_accuracy
    hm = numerator / denominator
    return hm

print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")

The approach resulted in no improvement over the zero-shot baseline, and only a bit of flection in performance results compared to zero-shot, probably also due to the overlapping of names for flowers and colors(violet, lilac, ...). Moreover the infernce time doubled compared to the baseline, due to the double use of clip. This issue could be solved by using a different model, or training one for color classification, but the idea was abandoned due to the fact that no improvement was fonud.

## CoOp TODO be more precise and explain the method and add reference

As next step, we decided to develop CoOp, following their paper, to then look for new original ideas/modifications that could lead us to better results. This approach aims at training a fixed amount of context token vectors that are fed to clip along with the classes names, allowing for a degree of freedom to push the right classes closer to the image vector represenations, to improve performance. During training, it minimizes prediction errors by using the cross-entropy loss with respect to the learnable context vectors while keeping the entire pre-trained parameters fixed. The gradients can be back-propagated all the way through the text encoder, distilling the rich knowledge encoded in the parameters for learning task-relevant context.

In [ ]:
clip_model.eval()  # We won't fine-tune CLIP
for param in clip_model.parameters():
    param.requires_grad = False

# Get the image and text encoders
image_encoder = clip_model.visual
text_encoder = clip_model.encode_text  # Used in inference mode only

In [ ]:
class CoOpPromptLearner(nn.Module):
    def __init__(self, class_names, clip_model, n_ctx=16):
      super().__init__()
      self.class_names = class_names
      self.n_cls = len(class_names)
      self.n_ctx = n_ctx
      self.clip_model = clip_model
      self.tokenizer = clip.tokenize

      # CLIP parameters
      dtype = clip_model.dtype
      ctx_dim = clip_model.ln_final.weight.shape[0]
      self.ctx_dim = ctx_dim
      # Get the device from the clip_model
      self.device = next(clip_model.parameters()).device


      # Random context initialization
      ctx_vectors = torch.empty(n_ctx, ctx_dim, dtype=dtype)
      nn.init.normal_(ctx_vectors, std=0.02)
      prompt_prefix = " ".join(["X"] * n_ctx)

      self.ctx = nn.Parameter(ctx_vectors)  # shared learnable context, to be optimized

      # Use clip.tokenize directly as it doesn't have an encode method
      name_lens = [len(self.tokenizer(name)[0]) for name in class_names]
      prompts = [prompt_prefix + " " + name + "." for name in class_names]

      tokenized_prompts = torch.cat([self.tokenizer(prompt) for prompt in prompts]).to(self.device) # Move to device
      with torch.no_grad():
        embedding = clip_model.token_embedding(tokenized_prompts).type(dtype)

      self.register_buffer("token_prefix", embedding[:, :1, :]) # SOS
      self.register_buffer("token_suffix", embedding[:, 1 + n_ctx :, :]) # CLS, EOS

      self.tokenized_prompts = tokenized_prompts
      self.name_lens = name_lens


    def forward(self):

      prefix = self.token_prefix
      ctx = self.ctx
      if ctx.dim() == 2:
            ctx = ctx.unsqueeze(0).expand(self.n_cls, -1, -1)
      suffix = self.token_suffix

      prompts = torch.cat(
        [
          prefix,
          ctx,
          suffix,
        ],
        dim = 1,
      )

      return prompts

In [ ]:
class CoOpTextEncoder(nn.Module):
  def __init__(self, clip_model):
    super().__init__()
    self.transformer = clip_model.transformer
    self.positional_embedding = clip_model.positional_embedding
    self.ln_final = clip_model.ln_final
    self.text_projection = clip_model.text_projection
    self.dtype = clip_model.dtype

  def forward(self, prompts, tokenized_prompts):
    x = prompts + self.positional_embedding.type(self.dtype)
    x = x.permute(1,0,2)  # [seq_len, n_cls, embed_dim]
    x = self.transformer(x)
    x = x.permute(1,0,2)  # [n_cls, seq_len, embed_dim]
    x = self.ln_final(x).type(self.dtype)

    # Take features from EOS embedding
    x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)] @ self.text_projection

    return x

In [ ]:
class CoOpCLIP(nn.Module):
  def __init__(self, class_names, clip_model, n_ctx=16):
    super().__init__()
    self.prompt_learner = CoOpPromptLearner(class_names, clip_model, n_ctx)
    self.tokenized_prompts = self.prompt_learner.tokenized_prompts
    self.text_encoder = CoOpTextEncoder(clip_model)
    self.image_encoder = clip_model.visual
    self.logit_scale = clip_model.logit_scale
    self.dtype = clip_model.dtype


  def forward(self, images):

    # Encode images
    image_features = self.image_encoder(images.type(self.dtype))

    # Encode text
    prompts = self.prompt_learner()
    tokenized_prompts = self.tokenized_prompts
    text_features = self.text_encoder(prompts, tokenized_prompts)

    # Normalize
    image_features = image_features / image_features.norm(dim=1, keepdim=True)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)

    # Compute similarity logits
    logit_scale = self.logit_scale.exp()
    logits = logit_scale * image_features @ text_features.t() # Corrected line

    return logits

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_function, device):
  model.train()
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    logits = model(images)
    loss = loss_function(logits, labels)
    loss.backward()
    optimizer.step()
    

    total_loss += loss.item() * images.size(0)
    total_correct += (logits.argmax(dim=1) == labels).sum().item()
    total_samples += images.size(0)

  scheduler.step()
  avg_loss = total_loss / total_samples
  accuracy = total_correct / total_samples
  return avg_loss, accuracy


def evaluate(model, dataloader, device):
  model.eval()
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for images, labels in dataloader:
      images = images.to(device)
      labels = labels.to(device)

      logits = model(images)
      total_correct += (logits.argmax(dim=1) == labels).sum().item()
      total_samples += images.size(0)

  accuracy = total_correct / total_samples
  return accuracy

In [ ]:
# Create properly remapped datasets for base classes
train_base_remapped = create_remapped_dataset(train_set, base_classes)
val_base_remapped = create_remapped_dataset(val_set, base_classes)
test_base_remapped = create_remapped_dataset(test_set, base_classes)

# Wrap with RemappedDataset to get correct labels
train_base_dataset = RemappedDataset(train_base_remapped)
val_base_dataset = RemappedDataset(val_base_remapped)
test_base_dataset = RemappedDataset(test_base_remapped)
base_class_names = [CLASS_NAMES[i] for i in base_classes]
print(f"Base class names: {base_class_names[:5]}...")  # Show first 5

In [ ]:
coop_batch_size = 32
coop_train_loader = torch.utils.data.DataLoader(
    train_base_dataset,
    batch_size=coop_batch_size,
    shuffle=True,
    num_workers=2
)
coop_val_loader = torch.utils.data.DataLoader(
    val_base_dataset,
    batch_size=coop_batch_size,
    shuffle=False,
    num_workers=2
)
coop_test_loader = torch.utils.data.DataLoader(
    test_base_dataset,
    batch_size=coop_batch_size,
    shuffle=False,
    num_workers=2
)

In [ ]:
# Initialize the CoOp model
coop_model = CoOpCLIP(base_class_names, clip_model, n_ctx=16).to(device)

# Set up optimizer - only optimize the context vectors
coop_optimizer = torch.optim.SGD([coop_model.prompt_learner.ctx], lr=0.002, momentum=0.9)

coop_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(coop_optimizer, T_max=200, eta_min=0.0001)

# Loss function
coop_loss_function = nn.CrossEntropyLoss()

# Training parameters
coop_num_epochs = 25
coop_best_val_acc = 0.0

print(f"Model initialized with {len(base_class_names)} base classes")
print(f"Context dimension: {coop_model.prompt_learner.ctx_dim}")
print(f"Number of context tokens: {coop_model.prompt_learner.n_ctx}")
print(f"Trainable parameters: {sum(p.numel() for p in coop_model.parameters() if p.requires_grad)}")

# Verify only context vectors are trainable
print("\nTrainable parameters:")
for name, param in coop_model.named_parameters():
    if param.requires_grad:
        print(f"  {name}: {param.shape}")

In [ ]:
# Training loop
print("Starting training...")
coop_training_history = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': []
}

for epoch in range(coop_num_epochs):
    print(f"\nEpoch {epoch+1}/{coop_num_epochs}")
    print("-" * 50)

    # Training phase
    train_loss, train_acc = train_one_epoch(coop_model, coop_train_loader, coop_optimizer, coop_scheduler, coop_loss_function, device)

    # Validation phase
    val_acc = evaluate(coop_model, coop_val_loader, device)

    # Save training history
    coop_training_history['train_loss'].append(train_loss)
    coop_training_history['train_acc'].append(train_acc)
    coop_training_history['val_acc'].append(val_acc)

    # Print metrics
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > coop_best_val_acc:
        coop_best_val_acc = val_acc
        print(f"New best validation accuracy: {coop_best_val_acc:.4f}")
        # Save model checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': coop_model.state_dict(),
            'optimizer_state_dict': coop_optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_coop_model.pth')

print(f"\nTraining completed!")
print(f"Best validation accuracy: {coop_best_val_acc:.4f}")

This approach allowed us to grasp the strength of the context tokens that surround the class' ones; since it tackles all classes at once, it can lead to expect also an improvement on the novel classes, caused by the idea that the learned context is general enough to work with them too; however CoOp greatly improves performance on base classes but performes poorly on unseen classes, even much worse than the Zero-shot baseline, making it highly unsuitable for our objective, due to the high deficit on novel classes that we would have to regain.

## CoCoOp 

If CoOp allowed to learn the context by learning a single representation acrossed all training data, CoCoOp provides a dual approach, learning to generate a perturbation conditioned by the current image along the initial learnable context, that influences the resulting text features for each single instance.
CoCoOp is implemented by instantiating a lightweight neural networks, called Meta-Net, on top of the M
context vectors, to generate for each input a conditional token (vector), which is then combined with the context vectors.


In [ ]:
_tokenizer = _Tokenizer()

In [ ]:
class CoCoOpTextEncoder(nn.Module):
  def __init__(self, clip_model):
    super().__init__()
    self.transformer = clip_model.transformer
    self.positional_embedding = clip_model.positional_embedding
    self.ln_final = clip_model.ln_final
    self.text_projection = clip_model.text_projection  # This was missing!
    self.dtype = clip_model.dtype

  def forward(self, prompts, tokenized_prompts):
    x = prompts + self.positional_embedding.type(self.dtype)
    x = x.permute(1,0,2)  # [seq_len, n_cls, embed_dim]
    x = self.transformer(x)
    x = x.permute(1,0,2)  # [n_cls, seq_len, embed_dim]
    x = self.ln_final(x).type(self.dtype)

    # Take features from EOS embedding
    x = x[torch.arange(x.shape[0]), tokenized_prompts.argmax(dim=-1)] @ self.text_projection

    return x

In [ ]:
class CoCoOpPromptLearner(nn.Module):
    def __init__(self, class_names, clip_model, n_ctx=4, ctx_init=None): # (self, cfg, classnames, clip_model):
        super().__init__()
        n_cls = len(class_names)
        # ctx_init = cfg.TRAINER.COCOOP.CTX_INIT # ???
        dtype = clip_model.dtype
        ctx_dim = clip_model.ln_final.weight.shape[0]
        vis_dim = clip_model.visual.output_dim
        clip_imsize = clip_model.visual.input_resolution
        # cfg_imsize = cfg.INPUT.SIZE[0] # ???
        # assert cfg_imsize == clip_imsize, f"cfg_imsize ({cfg_imsize}) must equal to clip_imsize ({clip_imsize})"
        cfg_imsize = clip_imsize
        if ctx_init:
            # use given words to initialize context vectors
            ctx_init = ctx_init.replace("_", " ")
            n_ctx = len(ctx_init.split(" "))
            prompt = clip.tokenize(ctx_init).to(clip_model.token_embedding.weight.device) # Move to device
            with torch.no_grad():
                embedding = clip_model.token_embedding(prompt).type(dtype)
            ctx_vectors = embedding[0, 1 : 1 + n_ctx, :]
            prompt_prefix = ctx_init
        else:
            # random initialization
            ctx_vectors = torch.empty(n_ctx, ctx_dim, dtype=dtype)
            nn.init.normal_(ctx_vectors, std=0.02)
            prompt_prefix = " ".join(["X"] * n_ctx)

        print(f'Initial context: "{prompt_prefix}"')
        print(f"Number of context words (tokens): {n_ctx}")

        # Learnable context vectors - these are the main parameters being optimized
        self.ctx = nn.Parameter(ctx_vectors)

        # Meta-network that generates conditional bias based on image features
        # This is the key innovation of CoCoOp - making prompts conditional on input images
        self.meta_net = nn.Sequential(OrderedDict([
            ("linear1", nn.Linear(vis_dim, vis_dim // 16)),
            ("relu", nn.ReLU(inplace=True)),
            ("linear2", nn.Linear(vis_dim // 16, ctx_dim))
        ]))
        self.meta_net.to(dtype)

        # Use half precision
        # if cfg.TRAINER.COCOOP.PREC == "fp16":
        self.meta_net.half()

        # Process class names and create tokenized prompts
        classnames = [name.replace("_", " ") for name in class_names]
        name_lens = [len(_tokenizer.encode(name)) for name in class_names]
        # Create prompt template: "[CTX] [CLASS]."
        prompts = [prompt_prefix + " " + name + "." for name in class_names]

        # Tokenize all prompts for all classes and move to the correct device
        tokenized_prompts = torch.cat([clip.tokenize(p).to(clip_model.token_embedding.weight.device) for p in prompts])  # (n_cls, n_tkn)
        with torch.no_grad():
            embedding = clip_model.token_embedding(tokenized_prompts).type(dtype)

        # These token vectors will be saved when in save_model(),
        # but they should be ignored in load_model() as we want to use
        # those computed using the current class names
        # Store fixed parts of prompts: start-of-sentence token
        self.register_buffer("token_prefix", embedding[:, :1, :])  # SOS
        # Store fixed parts of prompts: class name and end-of-sentence tokens
        self.register_buffer("token_suffix", embedding[:, 1 + n_ctx :, :])  # CLS, EOS

        self.n_cls = n_cls
        self.n_ctx = n_ctx
        self.tokenized_prompts = tokenized_prompts  # torch.Tensor
        self.name_lens = name_lens

    # Construct complete prompts by concatenating prefix, context, and suffix
    def construct_prompts(self, ctx, prefix, suffix, label=None):
        # dim0 is either batch_size (during training) or n_cls (during testing)
        # ctx: context tokens, with shape of (dim0, n_ctx, ctx_dim)
        # prefix: the sos token, with shape of (n_cls, 1, ctx_dim)
        # suffix: remaining tokens, with shape of (n_cls, *, ctx_dim)

        # If specific labels are provided, select corresponding prefix/suffix
        if label is not None:
            prefix = prefix[label]
            suffix = suffix[label]

        # Concatenate all prompt components
        prompts = torch.cat(
            [
                prefix,  # (dim0, 1, dim)
                ctx,     # (dim0, n_ctx, dim)
                suffix,  # (dim0, *, dim)
            ],
            dim=1,
        )

        return prompts

    def forward(self, im_features):
        prefix = self.token_prefix
        suffix = self.token_suffix
        # Base context vectors (shared across all images)
        ctx = self.ctx                     # (n_ctx, ctx_dim)
        # Generate image-conditional bias based on image features
        im_features = im_features.to(self.ctx.dtype)
        bias = self.meta_net(im_features)  # (batch, ctx_dim)
        # Reshape bias to match context dimensions
        bias = bias.unsqueeze(1)           # (batch, 1, ctx_dim)
        # Expand base context to batch dimension
        ctx = ctx.unsqueeze(0)             # (1, n_ctx, ctx_dim)
        # Apply conditional shift: this is where CoCoOp differs from CoOp
        ctx_shifted = ctx + bias           # (batch, n_ctx, ctx_dim)

        # Use instance-conditioned context tokens for all classes
        # For each image in the batch, create prompts for all classes
        prompts = []
        for ctx_shifted_i in ctx_shifted:
            # Expand the shifted context for this image to all classes
            ctx_i = ctx_shifted_i.unsqueeze(0).expand(self.n_cls, -1, -1)
            # Construct complete prompts for all classes
            pts_i = self.construct_prompts(ctx_i, prefix, suffix)  # (n_cls, n_tkn, ctx_dim)
            prompts.append(pts_i)
        # Stack prompts for all images in batch
        prompts = torch.stack(prompts)

        return prompts

In [ ]:
# Main model that combines image encoder, prompt learner, and text encoder
# Text encoder is the same as CoOp
class CoCoOpCustomCLIP(nn.Module):
    def __init__(self, classnames, clip_model, n_ctx, ctx_init=None):
        super().__init__()
        # Initialize the conditional prompt learner
        self.prompt_learner = CoCoOpPromptLearner(classnames, clip_model, n_ctx, ctx_init=ctx_init)
        self.tokenized_prompts = self.prompt_learner.tokenized_prompts
        # Use pre-trained CLIP components
        self.image_encoder = clip_model.visual
        self.text_encoder = CoCoOpTextEncoder(clip_model)
        self.logit_scale = clip_model.logit_scale
        self.dtype = clip_model.dtype

    def forward(self, image, label=None):
        tokenized_prompts = self.tokenized_prompts
        # Get learnable temperature parameter from CLIP
        logit_scale = self.logit_scale.exp()

        # Encode images using CLIP's visual encoder
        image_features = self.image_encoder(image.type(self.dtype))
        # Normalize image features
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Generate conditional prompts based on image features
        prompts = self.prompt_learner(image_features)

        # Compute logits for each image-prompt pair
        logits = []
        for pts_i, imf_i in zip(prompts, image_features):
            # Encode text prompts for current image
            text_features = self.text_encoder(pts_i, tokenized_prompts)
            # Normalize text features
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            # Compute similarity scores (logits) between image and all class prompts
            l_i = logit_scale * imf_i @ text_features.t()
            logits.append(l_i)
        # Stack logits for all images
        logits = torch.stack(logits)

        # During training, return cross-entropy loss
        if self.prompt_learner.training:
            return F.cross_entropy(logits, label)

        # During inference, return logits
        return logits

In [ ]:
# Create data loaders
batch_size = 1
train_loader = torch.utils.data.DataLoader(
    train_base_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)
val_loader = torch.utils.data.DataLoader(
    val_base_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_base_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

In [ ]:
# Building model
cocoop_model = CoCoOpCustomCLIP(classnames=base_class_names, clip_model=clip_model, n_ctx=4, ctx_init = "A photo of a")
print("Turning off gradients in both the image and the text encoder")
# Only train the prompt learner, freeze CLIP components
name_to_update = "prompt_learner"

for name, param in cocoop_model.named_parameters():
    if name_to_update not in name:
        param.requires_grad_(False)
# Double check which parameters will be updated (maybe delete)
enabled = set()
for name, param in cocoop_model.named_parameters():
    if param.requires_grad:
        enabled.add(name)
print(f"Parameters to be updated: {enabled}")

cocoop_model.to(device)

# Same parameters/set up of coop
# # Set up optimizer
optimizer = torch.optim.SGD(
    filter(lambda p: p.requires_grad, cocoop_model.parameters()),
    lr=0.002,
    momentum=0.9,
    weight_decay=0.0,
    dampening=0.0,
    nesterov=False
)

# Scheduler - needs to be corrected based on your config
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10,           # MAX_EPOCH: 10 (not 200)
    eta_min=1e-5        # WARMUP_CONS_LR: 1e-5 (not 0.0001)
)
# # Loss function
loss_function = nn.CrossEntropyLoss()
# # Training parameters
num_epochs = 10
cocoop_best_val_acc = 0.0

print(f"Model initialized with {len(base_class_names)} base classes")
print(f"Context dimension: {cocoop_model.prompt_learner.n_ctx}")
print(f"Number of context tokens: {cocoop_model.prompt_learner.n_ctx}")
print(f"Trainable parameters: {sum(p.numel() for p in cocoop_model.parameters() if p.requires_grad)}")

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_function, device):
  model.train()
  total_loss = 0.0
  total_correct = 0
  total_samples = 0

  for images, labels in dataloader:
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    loss = model(images, labels)
    # logits = model(images)
    # loss = loss_function(logits, labels)
    loss.backward()
    optimizer.step()


    with torch.no_grad():
        model.eval()  # Temporarily switch to eval mode for logits
        logits = model(images)  # Get logits for accuracy calculation
        model.train()  # Switch back to training mode
    total_loss += loss.item() * images.size(0)
    total_correct += (logits.argmax(dim=1) == labels).sum().item()
    total_samples += images.size(0)

  scheduler.step()
  avg_loss = total_loss / total_samples
  accuracy = total_correct / total_samples
  return avg_loss, accuracy


def evaluate(model, dataloader, device):
  model.eval()
  total_correct = 0
  total_samples = 0

  with torch.no_grad():
    for images, labels in dataloader:
      images = images.to(device)
      labels = labels.to(device)

      logits = model(images)
      total_correct += (logits.argmax(dim=1) == labels).sum().item()
      total_samples += images.size(0)

  accuracy = total_correct / total_samples
  return accuracy

In [ ]:
# Training loop
print("Starting training...")
cocoop_training_history = {
    'train_loss': [],
    'train_acc': [],
    'val_acc': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    # Training phase
    train_loss, train_acc = train_one_epoch(cocoop_model, train_loader, optimizer, scheduler, loss_function, device)

    # Validation phase
    val_acc = evaluate(cocoop_model, val_loader, device)

    # Save training history
    cocoop_training_history['train_loss'].append(train_loss)
    cocoop_training_history['train_acc'].append(train_acc)
    cocoop_training_history['val_acc'].append(val_acc)

    # Print metrics
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

    # Save best model
    if val_acc > cocoop_best_val_acc:
        cocoop_best_val_acc = val_acc
        print(f"New best validation accuracy: {cocoop_best_val_acc:.4f}")

        torch.save({
            'epoch': epoch,
            'model_state_dict': cocoop_model.state_dict(),  # Full state dict
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'training_history': cocoop_training_history,
            'best_val_acc': cocoop_best_val_acc
        }, 'best_cocoop_model.pth')
print(f"\nTraining completed!")
print(f"Best validation accuracy: {cocoop_best_val_acc:.4f}")
checkpoint = torch.load('best_cocoop_model.pth')
cocoop_model.load_state_dict(checkpoint['model_state_dict'])

test_acc = evaluate(cocoop_model, test_loader, device)
print(f"Test Accuracy on Base Classes: {test_acc:.4f}")

In [ ]:
# Now let's evaluate on novel classes to test generalization
print("\n" + "="*60)
print("EVALUATING ON NOVEL CLASSES")
print("="*60)

# Create novel class datasets with remapped labels
test_novel_remapped = create_remapped_dataset(test_set, novel_classes)
test_novel_dataset = RemappedDataset(test_novel_remapped)

# # Get novel class names
novel_class_names = [CLASS_NAMES[i] for i in novel_classes]
print(f"Novel classes: {len(novel_classes)}")
print(f"Novel test samples: {len(test_novel_dataset)}")
print(f"Novel class names: {novel_class_names[:5]}...")

# Create a new model for novel classes (same learned context, different class names)
novel_cocoop_model = CoCoOpCustomCLIP(novel_class_names, clip_model, n_ctx=4, ctx_init="A photo of a").to(device)

# Load the learned context from the trained model
novel_cocoop_model.prompt_learner.ctx.data = cocoop_model.prompt_learner.ctx.data.clone()

# Create data loader for novel classes
test_novel_loader = torch.utils.data.DataLoader(
    test_novel_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

# Evaluate on novel classes
novel_test_acc = evaluate(novel_cocoop_model, test_novel_loader, device)
print(f"\nTest Accuracy on Novel Classes: {novel_test_acc:.4f}")

# Compare base vs novel performance
print(f"\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
print(f"Base Classes Test Accuracy: {test_acc:.4f}")
print(f"Novel Classes Test Accuracy: {novel_test_acc:.4f}")
print(f"Average Accuracy: {(test_acc + novel_test_acc) / 2:.4f}")

In contrast with what can be one's initial beliefs, CoCoOp provided a slightly worse improvement on base classes compared to CoOp, while yielding a better performance on the unseen classes, striking a balance between the baseline and CoOp, but greatly improving the harmonic mean.
This is because instance-conditional context can generalize better because it shifts the focus away from a specific set of classes—for reducing overfitting—to each input instance, and hence to the entire task.

## MoCoOp

Our research for literature that allowed to improve base performance without drawbacks on the novel accuracy led us to MoCoOp: Mixture of Prompt Learning for Vision Language Models. Their model trains a gate-neural network, that selects the best expert among a given set. The expert consists on a specific prompt on which CoOp is applied, while using a custom loss for training. Moreover, the two most suitable experts are selected, and their embeddings are combined; the result is then used for classification.

In [ ]:
# code here

The model satisfies the performance constraints, improving base classes' performances while maintaining good results on the novel ones.

## KgCoOp

Implementing and analyzing MoCoOp sparked the idea that the performance result achieved by the model could be mostly attributed to the custom loss, which penalized the model the more it learned a representation that was shifting away from the original context feature. Therefore we decided to apply this idea to the original CoOp approach, without using multiple experts, having to train a selector, and merge together different inference results for the best experts, creating this way a lighter and more essential pipeline.

In [ ]:
# code here

Although KgCoOp exhibits really good results on base classes, two main issues refrained us from keeping it as our final main solution:
- Performance on novel classes was still affected by slightly degradation.
- Further research led us to the finding of KgCoOp, a paper that already explores the subject at matter, deeming our implementation not novel.

### Performance in-depth analysis - class name misalignment discovery
With no significant results at hand, we opted for analyzing the performance results across all models, hoping to find a lead for an area/feature in which the models were performing poorly. Calculating the accuracy per class showed that all models, zero-shot included, were able to perform almost error-free predictions for some classes, while they were not able to identify not even a single element, or barely a couple for others.

In [ ]:
# code here

We then proceeded by analyzing a couple of the problematic classes: with little research through wikipedia, we found other alternative names that could be used to identify the same species; as soon as we tried a couple of them we observed major improvements in the selected classes.

In [ ]:
# code here - single example of class, with wiki screenshot/link

## AKA Clip

The great improvements in the few manually analyzed classes prompted at the fact that CLIP, even in it's zero shot form, was already able to recognize and correcly classify the different flowers, but was doing so while using different names, that had a slightly different representation compared to the original ones.
To address the issue we implemented a slightly modified version of zero shot: we first collected multiple names for all classes in the dataset. Then all the different aliases were used concurrently in the classification phase, and only the best one was selected. The selected one was then mapped to the original label and compared with the ground truth.

In [ ]:
CLASS_NAMES = [['pink primrose', 'pink evening primrose'], ['hard-leaved pocket orchid', 'silver slipper orchid'],
               ['canterbury bells', 'campanula medium'], ['sweet pea', 'lathyrus odoratus'], ['english marigold', 'calendula'],
               ['tiger lily'], ['moon orchid'], ['bird of paradise'], ['monkshood'], ['globe thistle'], ['snapdragon'],
               ["colt's foot", 'coltsfoot'], ['king protea'], ['spear thistle'], ['yellow iris'], ['globe-flower', 'trollius'],
               ['purple coneflower'], ['peruvian lily'], ['balloon flower', 'chinese bellflower'], ['giant white arum lily'],
               ['fire lily'], ['pincushion flower'], ['fritillary'], ['red ginger'], ['grape hyacinth'], ['corn poppy'],
               ['prince of wales feathers', 'amaranthus hypochondriacus'], ['stemless gentian'], ['artichoke', 'globe artichoke'],
               ['sweet william'], ['carnation'], ['garden phlox'], ['love in the mist', 'love-in-a-mist'],
               ['mexican aster', 'garden cosmos'], ['alpine sea holly'], ['ruby-lipped cattleya'],
               ['cape flower', 'japanese spider lily'], ['great masterwort', 'astrantia major'],
               ['siam tulip', 'curcuma alismatifolia'], ['lenten rose'], ['barbeton daisy', 'gerbera jamesonii'], ['daffodil'],
               ['sword lily', 'marsh gladiolus'], ['poinsettia'], ['bolero deep blue', 'eustoma exaltatum'],
               ['wallflower', 'erysimum'], ['marigold'], ['buttercup'], ['oxeye daisy'], ['common dandelion'], ['petunia'],
               ['wild pansy'], ['primula'], ['sunflower'], ['lilac hibiscus'], ['bishop of llandaff', 'dahlia bishop of llandaff'],
               ['gaura'], ['geranium'], ['orange dahlia'], ['pink-yellow dahlia', 'pink and yellow dahlia'], ['cautleya spicata'],
               ['japanese anemone'], ['black-eyed susan'], ['silverbush', 'shrubby bindweed'], ['californian poppy'],
               ['osteospermum'], ['spring crocus'], ['bearded iris'], ['windflower', 'wood anemone'], ['tree poppy'], ['gazania'],
               ['azalea'], ['water lily'], ['rose'], ['thorn apple', 'jimsonweed'], ['morning glory', 'morning-glory'],
               ['passion flower'], ['lotus'], ['toad lily'], ['anthurium'], ['frangipani'], ['clematis'], ['hibiscus'],
               ['columbine'], ['desert-rose'], ['tree mallow'], ['magnolia'], ['cyclamen'], ['watercress', 'nasturtium'],
               ['canna lily'], ['hippeastrum'], ['bee balm'], ['ball moss', 'tillandsia cyanea'], ['foxglove'], ['bougainvillea'],
               ['camellia'], ['mallow', 'abutilon'], ['mexican petunia', 'ruellia'], ['bromelia'], ['blanket flower'],
               ['trumpet creeper', 'campsis radicans'], ['blackberry lily']]

In [ ]:
@torch.no_grad() # we don't want gradients
def eval_aka_clip(model: nn.Module,
         dataset: torch.utils.data.Dataset,
         categories: list[int],
         batch_size: int,
         device: str,
         label=""):
    model.eval()

    # Remap labels into a contiguous set starting from zero
    true_label_to_index = {cat: idx for idx, cat in enumerate(categories)}
    log = pd.DataFrame(columns=["true_label", "predicted_label", "correct"])
    prompts = []
    map_alias_index_to_index_label = []
    map_alias_index_to_alias_label = []
    for c in categories:
        for alias in CLASS_NAMES[c]:
            prompts.append(f"a photo of a {alias}, a type of flower.")
            map_alias_index_to_index_label.append(true_label_to_index[c])
            map_alias_index_to_alias_label.append(alias)
    text_inputs = clip.tokenize(prompts).to(device)
    text_features = model.encode_text(text_inputs)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    correct_predictions = 0
    for image, target in tqdm(dataloader, desc=label):
        true_labels = [t.item() for t in target]
        target = torch.Tensor([true_label_to_index[t.item()] for t in target]).long()

        image = image.to(device)
        target = target.to(device)

        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        
        predicted_class = (image_features @ text_features.T).argmax(dim=-1)
        predicted_labels = [map_alias_index_to_alias_label[p.item()] for p in predicted_class]
        predicted_class = torch.Tensor([map_alias_index_to_index_label[p.item()] for p in predicted_class]).long().to(device)
        prediction_correctness = (predicted_class == target).tolist()
        batch_data = []
        for i, correct in enumerate(prediction_correctness):
            batch_data.append({"true_label": true_labels[i],
                              "predicted_label": predicted_labels[i],
                              "correct": correct})
        
        batch_df = pd.DataFrame(batch_data)
        log = pd.concat([log, batch_df], ignore_index=True)
        correct_predictions += sum(prediction_correctness)
    accuracy = correct_predictions / len(dataset)
    return accuracy, log

In [ ]:
base_accuracy, base_log = eval_aka_clip(model=clip_model, dataset=test_base, categories=base_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Base Classes")
novel_accuracy, novel_log = eval_aka_clip(model=clip_model, dataset=test_novel, categories=novel_classes, batch_size=128, device=device, label="🧠 Zero-shot evaluation on Novel Classes")
print()
print(f"🔍 Base classes accuracy: {base_accuracy*100:.2f}%")
print(f"🔍 Novel classes accuracy: {novel_accuracy*100:.2f}%")
print(f"🔍 Harmonic Mean: {harmonic_mean(base_accuracy, novel_accuracy)*100:.2f}%")

In [ ]:
map_alias_to_label_index = {}
for index in range(len(CLASS_NAMES)):
    for alias in CLASS_NAMES[index]:
        map_alias_to_label_index[alias] = index
base_classes_sample_counts = base_log['true_label'].value_counts().sort_index()
novel_classes_sample_counts = novel_log['true_label'].value_counts().sort_index()

# Concatenate base and novel counts
all_classes_sample_counts = pd.concat([base_classes_sample_counts, novel_classes_sample_counts]).sort_index()

# Merge base and novel logs
combined_log = pd.concat([base_log, novel_log], ignore_index=True)

# Compute correct predictions for each alias
alias_correct_predictions = combined_log[combined_log['correct'] == True]['predicted_label'].value_counts()

# Create metrics dictionary for each alias
alias_metrics = {}
for alias in map_alias_to_label_index.keys():
    class_index = map_alias_to_label_index[alias]
    
    # Get correct predictions for this alias
    correct_preds = alias_correct_predictions.get(alias, 0)
    
    # Get total samples for this class
    total_samples = all_classes_sample_counts.get(class_index, 0)
    
    # Get total predictions for this alias (correct + incorrect)
    total_predictions = combined_log[combined_log['predicted_label'] == alias].shape[0]
    
    # Calculate precision (correct predictions / total predictions for this alias)
    precision = correct_preds / total_predictions if total_predictions > 0 else 0.0
    recall = correct_preds / total_samples if total_samples > 0 else 0.0
    
    alias_metrics[alias] = {
        'correct_predictions': correct_preds,
        'total_class_samples': total_samples,
        'total_predictions': total_predictions,
        'precision': precision,
        'recall': recall
    }
alias_metrics_df = pd.DataFrame.from_dict(alias_metrics, orient='index')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(alias_metrics_df)

In [ ]:
# Count correct predictions for each class
base_correct_predictions_counts = base_log[base_log['correct'] == True]['true_label'].value_counts().sort_index()
novel_correct_predictions_counts = novel_log[novel_log['correct'] == True]['true_label'].value_counts().sort_index()

# Concatenate base and novel correct predictions
all_classes_correct_predictions_counts = pd.concat([base_correct_predictions_counts, novel_correct_predictions_counts]).sort_index()

# Calculate accuracy for each class
class_accuracies = {}
for class_id in range(len(CLASS_NAMES)):
    correct_preds = all_classes_correct_predictions_counts.get(class_id, 0)
    total_samples = all_classes_sample_counts.get(class_id, 0)
    recall = correct_preds / total_samples if total_samples > 0 else 0.0
    
    class_accuracies[class_id] = {
        'class_name': CLASS_NAMES[class_id][0],  # Primary name for the class
        'all_aliases': CLASS_NAMES[class_id],    # All aliases for this class
        'correct_predictions': correct_preds,
        'total_samples': total_samples,
        'recall': recall,
    }

# Convert to DataFrame
class_accuracies_df = pd.DataFrame.from_dict(class_accuracies, orient='index')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(class_accuracies_df)

In [ ]:
# Convert predicted aliases to class indices for base_log
base_log_with_pred_indices = base_log.copy()
base_log_with_pred_indices['predicted_label'] = base_log_with_pred_indices['predicted_label'].map(map_alias_to_label_index)
base_log_with_pred_indices['true_label'] = base_log_with_pred_indices['true_label'].astype(int)
base_log_with_pred_indices['predicted_label'] = base_log_with_pred_indices['predicted_label'].astype(int)

# Get true and predicted class indices
y_true = base_log_with_pred_indices['true_label'].values
y_pred = base_log_with_pred_indices['predicted_label'].values

# Create confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=base_classes)

# Create a more readable confusion matrix with class names
base_class_names = [CLASS_NAMES[i][0] for i in base_classes]  # Use primary name for each class

# Plot confusion matrix
plt.figure(figsize=(20, 16))
sns.heatmap(cm, 
            xticklabels=base_class_names, 
            yticklabels=base_class_names,
            annot=True, 
            fmt='d', 
            cmap='Blues',
            cbar_kws={'label': 'Number of Predictions'})

plt.title('Confusion Matrix for Base Classes (CLIP Zero-shot)', fontsize=16, pad=20)
plt.xlabel('Predicted Class', fontsize=14)
plt.ylabel('True Class', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print some statistics about the confusion matrix
print(f"📊 Confusion Matrix Statistics:")
print(f"Matrix shape: {cm.shape}")
print(f"Total predictions: {cm.sum()}")
print(f"Correct predictions (diagonal): {np.trace(cm)}")
print(f"Accuracy: {np.trace(cm) / cm.sum():.4f}")

# Find most confused classes (highest off-diagonal values)
cm_off_diagonal = cm.copy()
np.fill_diagonal(cm_off_diagonal, 0)
max_confusion_idx = np.unravel_index(np.argmax(cm_off_diagonal), cm_off_diagonal.shape)
max_confusion_count = cm_off_diagonal[max_confusion_idx]

true_class_idx = base_classes[max_confusion_idx[0]]
pred_class_idx = base_classes[max_confusion_idx[1]]

print(f"\n🔀 Most confused pair:")
print(f"True class: {CLASS_NAMES[true_class_idx][0]} (class {true_class_idx})")
print(f"Predicted as: {CLASS_NAMES[pred_class_idx][0]} (class {pred_class_idx})")
print(f"Confusion count: {max_confusion_count}")

In [ ]:
# Convert predicted aliases to class indices for novel_log
novel_log_with_pred_indices = novel_log.copy()
novel_log_with_pred_indices['predicted_label'] = novel_log_with_pred_indices['predicted_label'].map(map_alias_to_label_index)
novel_log_with_pred_indices['true_label'] = novel_log_with_pred_indices['true_label'].astype(int)
novel_log_with_pred_indices['predicted_label'] = novel_log_with_pred_indices['predicted_label'].astype(int)

# Get true and predicted class indices
y_true = novel_log_with_pred_indices['true_label'].values
y_pred = novel_log_with_pred_indices['predicted_label'].values

# Create confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=novel_classes)

# Create a more readable confusion matrix with class names
novel_class_names = [CLASS_NAMES[i][0] for i in novel_classes]  # Use primary name for each class

# Plot confusion matrix
plt.figure(figsize=(20, 16))
sns.heatmap(cm, 
            xticklabels=novel_class_names, 
            yticklabels=novel_class_names,
            annot=True, 
            fmt='d', 
            cmap='Blues',
            cbar_kws={'label': 'Number of Predictions'})

plt.title('Confusion Matrix for Novel Classes (CLIP Zero-shot)', fontsize=16, pad=20)
plt.xlabel('Predicted Class', fontsize=14)
plt.ylabel('True Class', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print some statistics about the confusion matrix
print(f"📊 Confusion Matrix Statistics:")
print(f"Matrix shape: {cm.shape}")
print(f"Total predictions: {cm.sum()}")
print(f"Correct predictions (diagonal): {np.trace(cm)}")
print(f"Accuracy: {np.trace(cm) / cm.sum():.4f}")

# Find most confused classes (highest off-diagonal values)
cm_off_diagonal = cm.copy()
np.fill_diagonal(cm_off_diagonal, 0)
max_confusion_idx = np.unravel_index(np.argmax(cm_off_diagonal), cm_off_diagonal.shape)
max_confusion_count = cm_off_diagonal[max_confusion_idx]

true_class_idx = novel_classes[max_confusion_idx[0]]
pred_class_idx = novel_classes[max_confusion_idx[1]]

print(f"\n🔀 Most confused pair:")
print(f"True class: {CLASS_NAMES[true_class_idx][0]} (class {true_class_idx})")
print(f"Predicted as: {CLASS_NAMES[pred_class_idx][0]} (class {pred_class_idx})")
print(f"Confusion count: {max_confusion_count}")

In [ ]:
# Summary statistics
total_aliases = len(alias_metrics)
aliases_with_predictions = sum(1 for metrics in alias_metrics.values() if metrics['total_predictions'] > 0)
avg_precision = sum(metrics['precision'] for metrics in alias_metrics.values()) / total_aliases

print(f"📊 Summary Statistics:")
print(f"Total aliases: {total_aliases}")
print(f"Aliases with at least one prediction: {aliases_with_predictions}")
print(f"Average precision across all aliases: {avg_precision:.4f}")

# Find best and worst performing aliases
best_alias = max(alias_metrics.items(), key=lambda x: x[1]['precision'])
worst_alias = min(alias_metrics.items(), key=lambda x: x[1]['precision'] if x[1]['total_predictions'] > 0 else float('inf'))

print(f"\n🏆 Best performing alias: {best_alias[0]} (precision: {best_alias[1]['precision']:.4f})")
print(f"🚫 Worst performing alias (with predictions): {worst_alias[0]} (precision: {worst_alias[1]['precision']:.4f})")

# Show aliases that were never predicted
never_predicted = [alias for alias, metrics in alias_metrics.items() if metrics['total_predictions'] == 0]
print(f"\n❌ Aliases never predicted: {len(never_predicted)} out of {total_aliases}")
if len(never_predicted) <= 10:
    print(f"Never predicted aliases: {never_predicted}")
else:
    print(f"First 10 never predicted aliases: {never_predicted[:10]}")

# Convert to DataFrame for easier analysis if needed
import pandas as pd
alias_metrics_df = pd.DataFrame.from_dict(alias_metrics, orient='index')
print(f"\n📈 Top 10 aliases by precision:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(alias_metrics_df.sort_values('precision', ascending=False).head(10))

The results show major improvements in both base classes, where names were selected among the best performing ones, and novel classes, were we limited our work at name collection with no selection, fueling the hypotesis that using the right terms to identify the wanted class outweights the improvements given by the selection of the best context tokens that come along.

## CoOp, CoCoOp, MoCoOp, KgCoOp performance with new selected names

To further experiment with the newfound names, we decided to quickly test the best names with the previously built implementation by simply substituting the original names with better performing ones.

In [ ]:
# code here

All approaches benefit from the name switching, and it looks like the different improvements stack together, leading to the best performances ever achieved.

# Conclusions and future work

Our work aims to highlight the centrality of the identity of the subject compared to the effectiveness of the right selection of context tokens; the model already exhibits great distinction among the different classes in its internal representation, but sometimes does use a different nomenclature to refer to the subject. A slight tuning of the classnames can be far more effective than any adjustment on the context tokens.
These adjustments could be also automated through the use of LLMs that can be used for the generation of alternative names, possibly rendering this approach suitable for also other datasets/tasks.